In [3]:
pip install kafka-python

Note: you may need to restart the kernel to use updated packages.


In [4]:
from kafka import KafkaConsumer, KafkaProducer
from kafka.admin import KafkaAdminClient
import json
import uuid

In [6]:
dlq_topic = 'dead_letter_queue'
bootstrap_servers = ['kafka:9092']

consumer = KafkaConsumer(
    dlq_topic,
    bootstrap_servers=bootstrap_servers,
    auto_offset_reset='earliest',
    enable_auto_commit=True
)

producer = KafkaProducer(
    bootstrap_servers=bootstrap_servers,
    key_serializer=lambda k: k.encode('utf-8') if isinstance(k, str) else k,
    value_serializer=lambda v: v.encode('utf-8') if isinstance(v, str) else v
)

admin_client = KafkaAdminClient(
    bootstrap_servers=bootstrap_servers,
    client_id='reprocess_dlq_messages'
)

messages = consumer.poll(timeout_ms=2000)

last_offsets = {} 

if messages:
    for topic_partition, records in messages.items():
        for message in records:
            try:
                message_decoded = message.value.decode('utf-8')
                json_obj = json.loads(json.loads(message_decoded))
                processing_topic = None
                if 'spO2' in json_obj:
                    processing_topic = json_obj["spO2"]["id"]
                elif 'bpSys' in json_obj:
                    processing_topic = json_obj["bpSys"]["id"]
                elif 'temp' in json_obj:
                    processing_topic = json_obj["temp"]["id"]
                elif 'ecg' in json_obj:
                    processing_topic = json_obj["ecg"]["id"]

                if processing_topic:
                    producer.send(processing_topic, key=str(uuid.uuid4()), value=message_decoded.encode('utf-8'))
                    producer.flush()
                    print(f"Message: {message_decoded} | Sent topic: {processing_topic}")
                
                last_offsets[topic_partition] = message.offset
            except Exception as e:
                print(f"Error to reprocess message: {e}")

    partitions_to_delete = {
        tp: last_offsets[tp] + 1 for tp in last_offsets
    }

    try:
        admin_client.delete_records(partitions_to_delete)
    except Exception as e:
        print(f"Error to clean DLQ: {e}")

consumer.close()
producer.close()
print("Closed connections")

Message: "{\"temp\": {\"id\": \"f50604b6-ce65-4905-abdb-a772293b5352\", \"name\": \"temperature\", \"device_type\": \"temp\", \"timestamp\": 1745272813, \"unit\": \"degrees\", \"value\": 36.19}}" | Sent topic: f50604b6-ce65-4905-abdb-a772293b5352
Message: "{\"temp\": {\"id\": \"81d26c14-fb01-42e4-8f4d-e4e7dd65ae9f\", \"name\": \"temperature\", \"device_type\": \"temp\", \"timestamp\": 1745272815, \"unit\": \"degrees\", \"value\": 36.66}}" | Sent topic: 81d26c14-fb01-42e4-8f4d-e4e7dd65ae9f
Message: "{\"ecg\": {\"id\": \"e1816935-9575-4f69-a5c8-1a52f96957e9\", \"name\": \"ECG\", \"device_type\": \"ecg\", \"timestamp\": 1745272817, \"unit\": \"bpm\", \"value\": 83}}" | Sent topic: e1816935-9575-4f69-a5c8-1a52f96957e9
Message: "{\"spO2\": {\"id\": \"9aa7c7c9-3ab6-4f2a-b091-72aefe1dda5d\", \"name\": \"oxygen saturation\", \"device_type\": \"spO2\", \"timestamp\": 1745272822, \"unit\": \"percentage\", \"value\": 81}}" | Sent topic: 9aa7c7c9-3ab6-4f2a-b091-72aefe1dda5d
Message: "{\"temp\": {\